# Train model IQA — đa nhãn 5 tác vụ (sigmoid heads)

Model IQA nhẹ (on-device) dự đoán **độ hữu dụng** cho 5 tác vụ, học từ **nhãn hợp nhất**
(`labels/fused/usability_labels.csv`). Mỗi ảnh thường chỉ có nhãn cho MỘT tác vụ → **masked BCE**.
Gold seed = tập kiểm thử sạch (loại khỏi train/val). Xem `paper/Methodology.md` §3.5–3.9.

Mẫu suy giảm LF6 (`image_id = base__axis__delta`) được áp `degrade()` on-the-fly trong Dataset; xem phần "LF6: toán tử suy giảm + metadata mẫu".

## Cài đặt

In [ ]:
%pip install -q torchvision onnx opencv-python-headless

## Cấu hình

In [ ]:
import platform
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torchvision

DRY_RUN = True
RUNNER = 'local'
SEED = 42
ARCH = 'mobilenet_v3_small'
INPUT_SIZES = [224, 320, 384]
EPOCHS = 40
BATCH = 32
LR = 0.0005
PATIENCE = 6
VAL_FRAC = 0.15
LABEL_MODE = 'hard'
EXPORT_ONNX = False

TASKS = ['1_maturity_evaluation', '2_foliar_disease', '3_trunk_disease', '4_crown_disease', '5_petiole']
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cpu'
CUDA_NAME = 'none'
if torch.cuda.is_available():
    DEVICE = 'cuda'
    CUDA_NAME = torch.cuda.get_device_name(0)
elif torch.backends.mps.is_available():
    DEVICE = 'mps'

print('python:', platform.python_version())
print('torch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('device:', DEVICE, '| cuda:', CUDA_NAME)
print('dry_run:', DRY_RUN, '| runner:', RUNNER, '| seed:', SEED)
print('arch:', ARCH, '| input_sizes:', INPUT_SIZES, '| label_mode:', LABEL_MODE)
print('epochs:', EPOCHS, '| batch:', BATCH, '| lr:', LR, '| patience:', PATIENCE, '| val_frac:', VAL_FRAC)
print('tasks:', TASKS)

## Đường dẫn + module dùng chung (`src/utils/lf_io.ipynb`)

In [ ]:
def build_root(runner):
    if runner == 'local':
        candidates = [Path.cwd(), Path.cwd().parent]
        for cand in candidates:
            if (cand / 'labels' / 'fused' / 'usability_labels.csv').exists():
                return cand
        raise SystemExit('Khong thay labels/fused/usability_labels.csv — chay fusion_label_model.ipynb truoc')
    if runner == 'kaggle':
        root = Path('/kaggle/input/coconut-iqa')
        if not (root / 'labels' / 'fused' / 'usability_labels.csv').exists():
            raise SystemExit('Khong thay /kaggle/input/coconut-iqa/labels/fused/usability_labels.csv')
        return root
    raise SystemExit("RUNNER phai la 'local' hoac 'kaggle'")

ROOT = build_root(RUNNER)
FUSED_CSV = ROOT / 'labels' / 'fused' / 'usability_labels.csv'
VOTES_DIR = ROOT / 'labels' / 'votes'
GOLD_CSV = ROOT / 'gold_seed' / 'gold_seed_labels.csv'
GOLD_MANIFEST = ROOT / 'gold_seed' / 'gold_seed_manifest.csv'
OUT_DIR = ROOT / 'labels' / 'iqa_model'
OUT_DIR.mkdir(parents=True, exist_ok=True)

UTILS = ROOT / 'src' / 'utils' / 'lf_io.ipynb'
if not UTILS.exists():
    raise SystemExit('Khong thay ' + str(UTILS))
get_ipython().run_line_magic('run', str(UTILS))

print('ROOT:', ROOT)
print('FUSED_CSV:', FUSED_CSV)
print('GOLD_CSV:', GOLD_CSV, '| ton tai:', GOLD_CSV.exists())
print('GOLD_MANIFEST:', GOLD_MANIFEST, '| ton tai:', GOLD_MANIFEST.exists())
print('OUT_DIR:', OUT_DIR)

## 1. Bảng nhãn: vector 5 chiều + mask theo ảnh

`y[k]` = nhãn hữu dụng tác vụ k; `mask[k]=1` nếu ảnh có nhãn cho tác vụ k. Gold seed tách làm test (đường dẫn từ manifest, phủ đủ 300).

In [ ]:
def image_paths_from_votes(votes_dir):
    mapping = {}
    src = {}
    for f in sorted(votes_dir.glob('lf*.csv')):
        d = pd.read_csv(f)
        if 'image_id' not in d.columns or 'path' not in d.columns:
            continue
        for r in d.itertuples():
            if isinstance(r.path, str):
                mapping[r.image_id] = r.path
                if 'source' in d.columns and isinstance(getattr(r, 'source', None), str):
                    src[r.image_id] = r.source
    return mapping, src


def original_id(image_id, source):
    text = str(image_id)
    if isinstance(source, str) and source.startswith('coconut-veirf'):
        return text.split('_jpg')[0]
    return text


TASK_INDEX = {}
for i, t in enumerate(TASKS):
    TASK_INDEX[t] = i

fused = pd.read_csv(FUSED_CSV)
id2path, id2src = image_paths_from_votes(VOTES_DIR)
gold = pd.read_csv(GOLD_CSV)
gold_ids = set(gold['image_id'].tolist())
if GOLD_MANIFEST.exists():
    gm = pd.read_csv(GOLD_MANIFEST)
    for r in gm.itertuples():
        if isinstance(r.path, str):
            id2path[r.image_id] = r.path

Y = {}
M = {}
for r in fused.itertuples():
    iid = r.image_id
    if iid not in id2path:
        continue
    if iid not in Y:
        Y[iid] = [0.0, 0.0, 0.0, 0.0, 0.0]
        M[iid] = [0, 0, 0, 0, 0]
    k = TASK_INDEX[r.task]
    if LABEL_MODE == 'soft':
        Y[iid][k] = float(r.prob_usable)
    else:
        Y[iid][k] = float(int(r.label))
    M[iid][k] = 1

all_ids = [i for i in Y.keys() if i not in gold_ids]
print('anh co nhan fusion (khong tinh gold):', len(all_ids))
print('anh gold (test) co path:', len(gold_ids & set(id2path.keys())))
cov = np.array([M[i] for i in all_ids]).sum(axis=0) if all_ids else np.zeros(5)
for t in TASKS:
    print('  nhan train tac vu', t, ':', int(cov[TASK_INDEX[t]]))

## LF6: toán tử suy giảm + metadata mẫu

Mẫu LF6 mang `image_id = base__axis__delta`, cột `path` trỏ **ảnh gốc sạch**. Dataset áp `degrade(img, axis, delta)` on-the-fly (toán tử copy từ `lf6_degradation.ipynb`) để ảnh khớp đúng lúc LF6 chấm. Xem `docs/LF6_Methodology.md`.

In [ ]:
# Copy nguyen tu notebooks/lf6_degradation.ipynb (giu suy giam khop dung luc LF6 cham).
import cv2
from PIL import Image


def _to_arr(img):
    return np.asarray(img.convert("RGB"), dtype=np.float32)


def _to_img(arr):
    clipped = np.clip(arr, 0, 255).astype(np.uint8)
    return Image.fromarray(clipped, "RGB")


def deg_blur(img, delta):
    if delta <= 0:
        return img
    arr = _to_arr(img)
    k = int(round(delta)) * 2 + 1
    out = cv2.GaussianBlur(
        src=arr,
        ksize=(k, k),
        sigmaX=float(delta),
    )
    return _to_img(out)


def deg_exposure(img, delta):
    if delta <= 0:
        return img
    factor = 2.0 ** (-delta)
    return _to_img(_to_arr(img) * factor)


def deg_resolution(img, delta):
    if delta <= 0:
        return img
    w, h = img.size
    f = max(0.02, 1.0 - float(delta))
    small_w = max(1, int(w * f))
    small_h = max(1, int(h * f))
    small = img.resize((small_w, small_h), Image.BILINEAR)
    return small.resize((w, h), Image.BILINEAR).convert("RGB")


def deg_occlusion(img, delta):
    if delta <= 0:
        return img
    arr = _to_arr(img)
    h = arr.shape[0]
    w = arr.shape[1]
    frac = min(0.95, float(delta))
    bh = int(round(h * frac ** 0.5))
    bw = int(round(w * frac ** 0.5))
    y0 = (h - bh) // 2
    x0 = (w - bw) // 2
    arr[y0:y0 + bh, x0:x0 + bw, :] = 0.0
    return _to_img(arr)


def deg_white_balance(img, delta):
    if delta <= 0:
        return img
    arr = _to_arr(img)
    arr[..., 0] *= (1.0 + float(delta))
    arr[..., 2] *= (1.0 - 0.5 * float(delta))
    return _to_img(arr)


OPERATORS = {}
OPERATORS["blur"] = deg_blur
OPERATORS["exposure"] = deg_exposure
OPERATORS["resolution"] = deg_resolution
OPERATORS["occlusion"] = deg_occlusion
OPERATORS["white_balance"] = deg_white_balance


def degrade(img, axis, delta):
    return OPERATORS[axis](img, delta)

In [ ]:
LF6_VOTES = ROOT / 'labels' / 'votes' / 'lf6_degradation.csv'
LF6_REQUIRED_COLS = ['image_id', 'path', 'axis', 'delta', 'original_id']

LF6_META = {}
if not LF6_VOTES.exists():
    print('LF6 votes chua co:', LF6_VOTES)
    print('-> train khong co nhan am LF6 (chay lf6_degradation.ipynb + export sang votes/)')
else:
    lf6 = pd.read_csv(LF6_VOTES)
    for col in LF6_REQUIRED_COLS:
        if col not in lf6.columns:
            raise SystemExit('LF6 votes thieu cot bat buoc: ' + col)
    for r in lf6.itertuples():
        iid = str(r.image_id)
        meta = {}
        meta['path'] = r.path
        meta['axis'] = str(r.axis)
        meta['delta'] = float(r.delta)
        meta['original_id'] = str(r.original_id)
        LF6_META[iid] = meta

print('LF6_META:', len(LF6_META), 'mau (image_id dang base__axis__delta, path tro anh goc)')

## 2. Group-split theo ảnh gốc (train/val); gold seed = test

In [ ]:
def group_key(iid):
    if iid in LF6_META:
        return LF6_META[iid]['original_id']
    src = id2src.get(iid, '')
    return original_id(iid, src)


shuffler = random.Random(SEED)
groups = {}
for iid in all_ids:
    g = group_key(iid)
    groups.setdefault(g, []).append(iid)

group_keys = sorted(groups.keys())
shuffler.shuffle(group_keys)
n_val = int(len(group_keys) * VAL_FRAC)
val_groups = set(group_keys[:n_val])

train_ids = []
val_ids = []
for g in group_keys:
    if g in val_groups:
        val_ids.extend(groups[g])
    else:
        train_ids.extend(groups[g])

test_ids = [i for i in gold_ids if i in id2path]
for iid in test_ids:
    if iid in LF6_META:
        raise SystemExit('LF6 sample lot vao test: ' + str(iid))
print('train:', len(train_ids), '| val:', len(val_ids), '| test(gold):', len(test_ids))

Y_gold = {}
for r in gold.itertuples():
    vec = []
    for t in TASKS:
        vec.append(float(int(getattr(r, t))))
    Y_gold[r.image_id] = vec

## 3. Dataset + DataLoader

In [ ]:
from PIL import Image, ImageOps
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def make_transform(input_size, train):
    steps = []
    if train:
        steps.append(transforms.RandomResizedCrop(input_size, scale=(0.7, 1.0)))
        steps.append(transforms.RandomHorizontalFlip())
    else:
        steps.append(transforms.Resize((input_size, input_size)))
    steps.append(transforms.ToTensor())
    steps.append(transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD))
    return transforms.Compose(steps)


class IqaDataset(Dataset):
    def __init__(self, ids, y_map, m_map, root, input_size, train):
        self.ids = ids
        self.y_map = y_map
        self.m_map = m_map
        self.root = root
        self.tf = make_transform(input_size, train)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        iid = self.ids[idx]
        if iid in LF6_META:
            meta = LF6_META[iid]
            src_path = self.root / meta['path']
            img = ImageOps.exif_transpose(Image.open(src_path)).convert('RGB')
            img = degrade(img, meta['axis'], meta['delta'])
        else:
            src_path = self.root / id2path[iid]
            img = ImageOps.exif_transpose(Image.open(src_path)).convert('RGB')
        x = self.tf(img)
        y = torch.tensor(self.y_map[iid], dtype=torch.float32)
        m = torch.tensor(self.m_map[iid], dtype=torch.float32)
        return x, y, m


def gold_mask_map(ids):
    m = {}
    for iid in ids:
        m[iid] = [1, 1, 1, 1, 1]
    return m

## Kiểm tra mẫu LF6

In [ ]:
lf6_train = [i for i in train_ids if i in LF6_META]
lf6_val = [i for i in val_ids if i in LF6_META]
print('LF6 trong train:', len(lf6_train), '| trong val:', len(lf6_val))

lf6_all = lf6_train + lf6_val
pos = 0
neg = 0
for iid in lf6_all:
    y = Y[iid]
    m = M[iid]
    for k in range(5):
        if m[k] == 1:
            if y[k] >= 0.5:
                pos = pos + 1
            else:
                neg = neg + 1
print('LF6 nhan 1:', pos, '| nhan 0:', neg)

n_check = min(2, len(lf6_all))
for j in range(n_check):
    iid = lf6_all[j]
    meta = LF6_META[iid]
    src_path = ROOT / meta['path']
    base_img = ImageOps.exif_transpose(Image.open(src_path)).convert('RGB')
    deg_img = degrade(base_img, meta['axis'], meta['delta'])
    a = np.asarray(base_img, dtype=np.float32)
    b = np.asarray(deg_img, dtype=np.float32)
    diff = float(np.abs(a - b).mean())
    print('kiem tra', iid, '| axis', meta['axis'], '| delta', meta['delta'], '| pixel_diff', round(diff, 4))

## 4. Model + masked weighted BCE

In [ ]:
import torch.nn as nn
from torchvision import models


def build_model(arch, n_out):
    if arch == 'mobilenet_v3_small':
        net = models.mobilenet_v3_small(weights='IMAGENET1K_V1')
        in_f = net.classifier[3].in_features
        net.classifier[3] = nn.Linear(in_f, n_out)
        return net
    if arch == 'mobilenet_v3_large':
        net = models.mobilenet_v3_large(weights='IMAGENET1K_V1')
        in_f = net.classifier[3].in_features
        net.classifier[3] = nn.Linear(in_f, n_out)
        return net
    if arch == 'efficientnet_b0':
        net = models.efficientnet_b0(weights='IMAGENET1K_V1')
        in_f = net.classifier[1].in_features
        net.classifier[1] = nn.Linear(in_f, n_out)
        return net
    raise SystemExit('ARCH khong ho tro: ' + arch)


def pos_weight_from_labels(ids, y_map, m_map):
    pos = np.zeros(5)
    neg = np.zeros(5)
    for iid in ids:
        y = y_map[iid]
        m = m_map[iid]
        for k in range(5):
            if m[k] == 1:
                if y[k] >= 0.5:
                    pos[k] = pos[k] + 1
                else:
                    neg[k] = neg[k] + 1
    w = np.ones(5)
    for k in range(5):
        if pos[k] > 0:
            w[k] = max(1.0, neg[k] / pos[k])
    return torch.tensor(w, dtype=torch.float32)


def masked_bce(logits, y, m, pos_weight):
    per = nn.functional.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight, reduction='none')
    per = per * m
    denom = m.sum()
    if denom <= 0:
        return per.sum() * 0.0
    return per.sum() / denom

## 5. Vòng train (early-stop trên masked val loss) — dò từng kích thước đầu vào

In [ ]:
def run_epoch(model, loader, pos_weight, optimizer, train):
    if train:
        model.train()
    else:
        model.eval()
    total = 0.0
    n = 0
    for x, y, m in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        m = m.to(DEVICE)
        if train:
            optimizer.zero_grad()
        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = masked_bce(logits, y, m, pos_weight.to(DEVICE))
            if train:
                loss.backward()
                optimizer.step()
        total = total + float(loss) * len(x)
        n = n + len(x)
    return total / max(1, n)


def train_one_size(input_size):
    pos_weight = pos_weight_from_labels(train_ids, Y, M)
    tr = DataLoader(IqaDataset(train_ids, Y, M, ROOT, input_size, True), batch_size=BATCH, shuffle=True, num_workers=2)
    va = DataLoader(IqaDataset(val_ids, Y, M, ROOT, input_size, False), batch_size=BATCH, shuffle=False, num_workers=2)
    model = build_model(ARCH, 5).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    best = float('inf')
    best_state = None
    bad = 0
    for ep in range(EPOCHS):
        tr_loss = run_epoch(model, tr, pos_weight, optimizer, True)
        va_loss = run_epoch(model, va, pos_weight, optimizer, False)
        print('size', input_size, '| epoch', ep, '| train', round(tr_loss, 4), '| val', round(va_loss, 4))
        if va_loss < best:
            best = va_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad = bad + 1
        if bad >= PATIENCE:
            print('early stop @ epoch', ep)
            break
    model.load_state_dict(best_state)
    ckpt = OUT_DIR / ('iqa_' + ARCH + '_' + str(input_size) + '.pt')
    torch.save(model.state_dict(), ckpt)
    print('luu:', ckpt, '| best val', round(best, 4))
    return model


trained = {}
if DRY_RUN:
    print('DRY_RUN = True -> bo qua train')
else:
    for size in INPUT_SIZES:
        trained[size] = train_one_size(size)

## 6. Đánh giá trên gold seed (calibrate τ trên val)

In [ ]:
from sklearn.metrics import f1_score, precision_recall_fscore_support, roc_auc_score
from sklearn.metrics import accuracy_score, hamming_loss


@torch.inference_mode()
def predict_probs(model, ids, y_map, m_map, input_size):
    loader = DataLoader(IqaDataset(ids, y_map, m_map, ROOT, input_size, False), batch_size=BATCH, shuffle=False, num_workers=2)
    model.eval()
    out = []
    for x, y, m in loader:
        x = x.to(DEVICE)
        logits = model(x)
        out.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(out, axis=0)


def calibrate_tau(probs_val, y_val, m_val):
    tau = np.full(5, 0.5)
    grid = np.linspace(0.05, 0.95, 19)
    for k in range(5):
        rows = m_val[:, k] == 1
        if rows.sum() == 0:
            continue
        yt = y_val[rows, k].astype(int)
        pv = probs_val[rows, k]
        best_f1 = -1.0
        for t in grid:
            f1 = f1_score(yt, (pv >= t).astype(int), zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                tau[k] = t
    return tau


def evaluate(model, input_size):
    y_val = np.array([Y[i] for i in val_ids])
    m_val = np.array([M[i] for i in val_ids])
    probs_val = predict_probs(model, val_ids, Y, M, input_size)
    tau = calibrate_tau(probs_val, y_val, m_val)
    print('tau (tren val):', dict(zip(TASKS, [round(float(t), 3) for t in tau])))

    gm = gold_mask_map(test_ids)
    probs = predict_probs(model, test_ids, Y_gold, gm, input_size)
    yt = np.array([Y_gold[i] for i in test_ids])
    preds = (probs >= tau).astype(int)

    rows = []
    for k, t in enumerate(TASKS):
        p, r, f, _ = precision_recall_fscore_support(yt[:, k], preds[:, k], labels=[1], average='binary', zero_division=0)
        try:
            auc = roc_auc_score(yt[:, k], probs[:, k])
        except ValueError:
            auc = float('nan')
        rows.append({'task': t, 'precision': round(p, 4), 'recall': round(r, 4), 'f1': round(f, 4), 'auc': round(auc, 4)})
    rep = pd.DataFrame(rows)
    macro_f1 = rep['f1'].mean()
    micro_f1 = f1_score(yt.ravel(), preds.ravel(), zero_division=0)
    subset_acc = accuracy_score(yt, preds)
    ham = hamming_loss(yt, preds)
    print(rep.to_string(index=False))
    print('macro-F1', round(macro_f1, 4), '| micro-F1', round(micro_f1, 4), '| subset-acc', round(subset_acc, 4), '| hamming', round(ham, 4))
    return rep, tau


if not DRY_RUN:
    for size in INPUT_SIZES:
        print('=== danh gia size', size, '===')
        evaluate(trained[size], size)

## 7. Kích thước / latency / xuất ONNX

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())


def measure_latency(model, input_size, n=20):
    model.eval()
    x = torch.randn(1, 3, input_size, input_size).to(DEVICE)
    import time
    with torch.inference_mode():
        for _ in range(3):
            model(x)
        t0 = time.time()
        for _ in range(n):
            model(x)
        dt = (time.time() - t0) / n
    return dt * 1000.0


if not DRY_RUN and EXPORT_ONNX:
    for size in INPUT_SIZES:
        model = trained[size]
        params = count_params(model)
        lat = measure_latency(model, size)
        onnx_path = OUT_DIR / ('iqa_' + ARCH + '_' + str(size) + '.onnx')
        dummy = torch.randn(1, 3, size, size).to(DEVICE)
        torch.onnx.export(model, dummy, str(onnx_path), input_names=['image'], output_names=['usability'], opset_version=17)
        mb = onnx_path.stat().st_size / 1e6
        print('size', size, '| params', params, '| latency_ms', round(lat, 2), '| onnx_MB', round(mb, 2), '->', onnx_path)

## Giới hạn

Nhãn train là nhãn yếu (fusion); mỗi ảnh thường chỉ biết 1 tác vụ → masked BCE. Mẫu LF6 (ảnh suy giảm) được áp `degrade()` on-the-fly trong Dataset (toán tử copy từ `lf6_degradation.ipynb`); nếu chưa có `labels/votes/lf6_degradation.csv` thì train chạy không có nhãn âm LF6. Gold seed nhỏ (300), lệch về "hữu dụng" → đọc kèm κ. `τ_k` calibrate trên val.